In [2]:


# import os
# import re
# import time
# import random
# import shutil
# import requests
# import pandas as pd
# from bs4 import BeautifulSoup
# from urllib.parse import urljoin, urlparse
# from concurrent.futures import ThreadPoolExecutor, as_completed

# # =====================================================
# # V2 CLEAN DATASET SCRAPER
# # Final CSV = NO URL columns
# # Only real doctor data + local images
# # =====================================================

# # ---------------- CONFIG ----------------
# BASE_URL = "https://www.doctorbangladesh.com"
# PAGE_URL = "https://www.doctorbangladesh.com/anesthesiologist-dhaka/"

# CSV_NAME = "anesthesiologist.csv"
# IMAGE_FOLDER = "anesthesiologist"

# MAX_WORKERS = 10
# TIMEOUT = 20
# RETRY = 3

# HEADERS = {
#     "User-Agent": "Mozilla/5.0"
# }

# os.makedirs(IMAGE_FOLDER, exist_ok=True)

# # =====================================================
# # HELPERS
# # =====================================================

# def clean_text(text):
#     if not text:
#         return None
#     text = re.sub(r'\s+', ' ', text).strip()
#     return text


# def safe_filename(text):
#     text = clean_text(text)
#     text = re.sub(r'[^a-zA-Z0-9\s-]', '', text)
#     text = text.replace(" ", "-").lower()
#     return text[:120]


# def request_html(url):
#     for _ in range(RETRY):
#         try:
#             r = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
#             r.raise_for_status()
#             return r.text
#         except:
#             time.sleep(random.uniform(1, 2))
#     return None


# def extract_text(tag):
#     if not tag:
#         return None
#     return clean_text(tag.get_text(" ", strip=True))


# def get_image_url(img):
#     if not img:
#         return None

#     for attr in ["src", "data-src", "data-lazy-src", "srcset"]:
#         val = img.get(attr)

#         if val:
#             if attr == "srcset":
#                 val = val.split(",")[0].split()[0]

#             return urljoin(BASE_URL, val.strip())

#     return None


# def file_ext(url):
#     ext = os.path.splitext(urlparse(url).path)[1].lower()
#     if ext in [".jpg", ".jpeg", ".png", ".webp"]:
#         return ext
#     return ".jpg"


# def download_image(url, doctor_name):
#     if not url:
#         return None

#     try:
#         filename = safe_filename(doctor_name) + file_ext(url)
#         filepath = os.path.join(IMAGE_FOLDER, filename)

#         if os.path.exists(filepath):
#             return filepath

#         r = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
#         r.raise_for_status()

#         with open(filepath, "wb") as f:
#             f.write(r.content)

#         return filepath

#     except:
#         return None


# # =====================================================
# # PROFILE PAGE SCRAPER
# # =====================================================

# # =====================================================
# # FIX ONLY THIS FUNCTION IN YOUR V2 SCRAPER
# # Replace old scrape_profile() with this improved version
# # =====================================================

# def scrape_profile(profile_url):

#     result = {
#         "extra_chambers": None,
#         "extra_addresses": None,
#         "extra_visiting_hours": None,
#         "appointments": None,
#         "about": None
#     }

#     html = request_html(profile_url)

#     if not html:
#         return result

#     soup = BeautifulSoup(html, "lxml")
#     entry = soup.find("div", class_="entry-content")

#     if not entry:
#         return result

#     chambers = []
#     addresses = []
#     hours = []
#     phones = []

#     for h2 in entry.find_all("h2"):

#         heading = extract_text(h2)

#         if not heading:
#             continue

#         # ABOUT SECTION
#         if "About" in heading:
#             p = h2.find_next_sibling("p")
#             if p:
#                 result["about"] = extract_text(p)
#             break

#         p = h2.find_next_sibling("p")
#         if not p:
#             continue

#         txt = extract_text(p)

#         # -------------------------------------------------
#         # HOSPITAL / PLACE NAME
#         # usually inside <strong><a>Hospital Name</a></strong>
#         # -------------------------------------------------
#         place_name = None

#         strong_tag = p.find("strong")
#         if strong_tag:
#             place_name = extract_text(strong_tag)

#         if not place_name:
#             place_name = heading

#         chambers.append(place_name)

#         # -------------------------------------------------
#         # ADDRESS
#         # -------------------------------------------------
#         m1 = re.search(r'Address:\s*(.*?)\s*Visiting Hour:', txt)

#         if m1:
#             pure_address = clean_text(m1.group(1))
#             full_address = f"[{place_name}] {pure_address}"
#             addresses.append(full_address)

#         # -------------------------------------------------
#         # VISITING HOUR
#         # -------------------------------------------------
#         m2 = re.search(r'Visiting Hour:\s*(.*?)\s*Appointment:', txt)

#         if m2:
#             hours.append(clean_text(m2.group(1)))

#         # -------------------------------------------------
#         # PHONE
#         # -------------------------------------------------
#         m3 = re.search(r'Appointment:\s*([+0-9]+)', txt)

#         if m3:
#             phones.append(clean_text(m3.group(1)))

#     result["extra_chambers"] = " | ".join(chambers) if chambers else None
#     result["extra_addresses"] = " | ".join(addresses) if addresses else None
#     result["extra_visiting_hours"] = " | ".join(hours) if hours else None
#     result["appointments"] = " | ".join(phones) if phones else None

#     return result


# # =====================================================
# # SCRAPE MAIN LISTING PAGE
# # =====================================================

# html = request_html(PAGE_URL)

# if not html:
#     print("Page failed to load")
#     exit()

# soup = BeautifulSoup(html, "lxml")
# doctors = soup.find_all("li", class_="doctor")

# print("Doctors Found:", len(doctors))

# rows = []

# for doctor in doctors:

#     title = extract_text(doctor.find("h3", class_="title"))

#     img = doctor.find("img")
#     image_url = get_image_url(img)

#     call = doctor.find("a", class_="call-now")
#     profile_url = call["href"] if call else None

#     degree = speciality = experience = designation = workplace = None
#     chamber = address = visiting_hour = None

#     doctor_info = doctor.find("ul", class_="doctor-info")

#     if doctor_info:
#         for li in doctor_info.find_all("li"):

#             li_title = li.get("title", "").strip()
#             txt = extract_text(li)

#             if li_title == "Degree":
#                 degree = txt
#             elif li_title == "Experiences":
#                 experience = txt
#             elif li_title == "Specialty":
#                 speciality = txt
#             elif li_title == "Designation":
#                 designation = txt
#             elif li_title == "Workplace":
#                 workplace = txt

#     chamber_info = doctor.find("ul", class_="chamber-info")

#     if chamber_info:
#         for li in chamber_info.find_all("li"):

#             li_title = li.get("title", "").strip()
#             txt = extract_text(li)

#             if li_title == "Chamber":
#                 chamber = txt
#             elif li_title == "Address":
#                 address = txt.replace("Address:", "").strip()
#             elif li_title == "Visiting Hour":
#                 visiting_hour = txt.replace("Visiting Hour:", "").strip()

#     rows.append({
#         "image_url": image_url,          # temp only
#         "profile_url": profile_url,     # temp only

#         "title": title,
#         "degree": degree,
#         "speciality": speciality,
#         "experience": experience,
#         "designation": designation,
#         "workplace": workplace,

#         "chamber": chamber,
#         "address": address,
#         "visiting_hour": visiting_hour
#     })


# # =====================================================
# # THREAD PROCESSING
# # =====================================================

# def process_row(row):

#     # profile scrape
#     extra = scrape_profile(row["profile_url"])
#     row.update(extra)

#     # download image locally
#     row["photo"] = download_image(row["image_url"], row["title"])

#     # remove urls completely
#     row.pop("image_url", None)
#     row.pop("profile_url", None)

#     return row


# final_rows = []

# with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

#     futures = [executor.submit(process_row, row) for row in rows]

#     for future in as_completed(futures):
#         final_rows.append(future.result())


# # =====================================================
# # DATAFRAME
# # =====================================================

# df = pd.DataFrame(final_rows)

# df = df.drop_duplicates(subset=["title"])
# df = df.fillna("null")

# df = df[
#     [
#         "photo",
#         "title",
#         "degree",
#         "speciality",
#         "experience",
#         "designation",
#         "workplace",
#         "chamber",
#         "address",
#         "visiting_hour",
#         "extra_chambers",
#         "extra_addresses",
#         "extra_visiting_hours",
#         "appointments",
#         "about"
#     ]
# ]

# # save
# df.to_csv(CSV_NAME, index=False, encoding="utf-8-sig")

# # zip images
# shutil.make_archive(IMAGE_FOLDER, "zip", IMAGE_FOLDER)

# print(df.head())
# print("Total Doctors:", len(df))
# print("CSV Saved:", CSV_NAME)
# print("ZIP Saved:", IMAGE_FOLDER + ".zip")

SyntaxError: invalid syntax (3868974557.py, line 1)

In [9]:
import os
import random
import re
import shutil
import time
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import urljoin, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup


# =============================================================================
# STEP 1: PROGRAM SETTINGS
# =============================================================================

# The main website.
BASE_URL = "https://www.doctorbangladesh.com"

# The page that contains the list of doctors.
PAGE_URL = "https://www.doctorbangladesh.com/cardiac-surgeon-dhaka/"

# The names of the files and folder that the program will create.
CSV_NAME = "cardiac-surgeon.csv"
IMAGE_FOLDER = "cardiac-surgeon"

# The program can work on five doctors at the same time.
MAX_WORKERS = 5

# Stop waiting if a website request takes more than 20 seconds.
TIMEOUT = 20

# Try a failed website request three times.
RETRY = 3

# This makes our request look like a normal browser request.
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,application/xml;q=0.9,"
        "image/avif,image/webp,*/*;q=0.8"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

# These are the columns that will appear in the final CSV file.
# Their order will stay exactly the same.
CSV_COLUMNS = [
    "photo",
    "title",
    "degree",
    "speciality",
    "experience",
    "designation",
    "workplace",
    "chamber",
    "address",
    "visiting_hour",
    "extra_chambers",
    "extra_addresses",
    "extra_visiting_hours",
    "appointments",
    "about",
]


# =============================================================================
# STEP 2: SMALL HELPER FUNCTIONS
# =============================================================================

def clean_text(text):
    """
    Remove extra spaces from text.

    Example:
        "  Hello    World  " becomes "Hello World"
    """
    if text is None:
        return None

    cleaned = re.sub(r"\s+", " ", str(text)).strip()

    if cleaned == "":
        return None

    return cleaned


def get_text(html_tag):
    """Take the visible text from one HTML tag."""
    if html_tag is None:
        return None

    return clean_text(html_tag.get_text(" ", strip=True))


def download_page(url):
    """
    Download one webpage.

    If the request fails, the program waits a little and tries again.
    """
    if not url:
        return None

    last_error = None

    for attempt_number in range(1, RETRY + 1):
        try:
            response = requests.get(
                url,
                headers=HEADERS,
                timeout=TIMEOUT,
            )

            # This creates an error for bad responses such as 404 or 500.
            response.raise_for_status()
            return response.text

        except requests.RequestException as error:
            last_error = error

            # Do not wait after the final attempt.
            if attempt_number < RETRY:
                time.sleep(random.uniform(1, 2))

    print(f"Could not download: {url}")
    print(f"Reason: {last_error}")
    return None


def make_safe_filename(doctor_name):
    """
    Turn a doctor's name into a safe filename.

    Example:
        "Dr. John Smith" becomes "dr-john-smith"
    """
    doctor_name = clean_text(doctor_name) or "doctor"

    # Keep only English letters, numbers, spaces, and hyphens.
    doctor_name = re.sub(r"[^a-zA-Z0-9\s-]", "", doctor_name)

    # Change spaces into one hyphen.
    doctor_name = re.sub(r"[\s-]+", "-", doctor_name)
    doctor_name = doctor_name.strip("-").lower()

    if doctor_name == "":
        doctor_name = "doctor"

    # Very long filenames can cause problems, so keep only 120 characters.
    return doctor_name[:120]


def find_image_url(image_tag):
    """Find the real image URL inside an HTML image tag."""
    if image_tag is None:
        return None

    possible_places = ["src", "data-src", "data-lazy-src", "srcset"]

    for place in possible_places:
        image_url = image_tag.get(place)

        if not image_url:
            continue

        # srcset can contain several images.
        # We use the first image from that list.
        if place == "srcset":
            image_url = image_url.split(",")[0].split()[0]

        # This also fixes URLs that start with / instead of https://.
        return urljoin(BASE_URL, image_url.strip())

    return None


def find_image_extension(image_url):
    """Find whether an image is JPG, PNG, or WEBP."""
    extension = os.path.splitext(urlparse(image_url).path)[1].lower()

    allowed_extensions = [".jpg", ".jpeg", ".png", ".webp"]

    if extension in allowed_extensions:
        return extension

    # Use JPG when the URL does not show a known image type.
    return ".jpg"


def download_image(image_url, doctor_name):
    """Download one doctor's picture and return its local file path."""
    if not image_url:
        return None

    filename = (
        make_safe_filename(doctor_name)
        + find_image_extension(image_url)
    )

    image_path = os.path.join(IMAGE_FOLDER, filename)

    # Do not download the image again if it already exists.
    if os.path.isfile(image_path) and os.path.getsize(image_path) > 0:
        return image_path

    try:
        response = requests.get(
            image_url,
            headers=HEADERS,
            timeout=TIMEOUT,
        )
        response.raise_for_status()

        # Sometimes an image URL can return an HTML error page.
        # This check stops us from saving that page as a picture.
        file_type = response.headers.get("Content-Type", "").lower()

        if file_type and not file_type.startswith("image/"):
            print(f"Skipped a non-image file: {image_url}")
            return None

        with open(image_path, "wb") as image_file:
            image_file.write(response.content)

        return image_path

    except (OSError, requests.RequestException) as error:
        print(f"Could not download picture for {doctor_name}: {error}")
        return None


# =============================================================================
# STEP 3: READ ONE DOCTOR'S PROFILE PAGE
# =============================================================================

def make_empty_profile():
    """Create empty values for information found on a profile page."""
    return {
        "extra_chambers": None,
        "extra_addresses": None,
        "extra_visiting_hours": None,
        "appointments": None,
        "about": None,
    }


def read_profile(profile_url):
    """Read the extra information from one doctor's profile page."""
    profile_data = make_empty_profile()
    html = download_page(profile_url)

    if not html:
        return profile_data

    page = BeautifulSoup(html, "lxml")
    main_part = page.find("div", class_="entry-content")

    if main_part is None:
        return profile_data

    chamber_names = []
    chamber_addresses = []
    visiting_times = []
    phone_numbers = []

    # Each H2 heading represents a section of the profile.
    for heading_tag in main_part.find_all("h2"):
        heading = get_text(heading_tag)

        if not heading:
            continue

        # The useful information is normally in the paragraph after the H2.
        paragraph = heading_tag.find_next_sibling("p")

        # Save the doctor's About section.
        if "about" in heading.lower():
            if paragraph is not None:
                profile_data["about"] = get_text(paragraph)
            continue

        if paragraph is None:
            continue

        full_text = get_text(paragraph)

        if not full_text:
            continue

        # The first bold text normally contains the chamber name.
        bold_text = paragraph.find("strong")
        chamber_name = get_text(bold_text) or heading
        chamber_names.append(chamber_name)

        # Find the words between "Address:" and "Visiting Hour:".
        address_result = re.search(
            r"Address:\s*(.*?)\s*Visiting Hour:",
            full_text,
            flags=re.IGNORECASE,
        )

        if address_result:
            address = clean_text(address_result.group(1))
            chamber_addresses.append(
                f"[{chamber_name}] {address}"
            )

        # Find the words between "Visiting Hour:" and "Appointment:".
        time_result = re.search(
            r"Visiting Hour:\s*(.*?)\s*Appointment:",
            full_text,
            flags=re.IGNORECASE,
        )

        if time_result:
            visiting_times.append(
                clean_text(time_result.group(1))
            )

        # Find the phone number after "Appointment:".
        phone_result = re.search(
            r"Appointment:\s*([+0-9][+0-9\s-]*)",
            full_text,
            flags=re.IGNORECASE,
        )

        if phone_result:
            phone_numbers.append(
                clean_text(phone_result.group(1))
            )

    # Join multiple values using | so they can fit inside one CSV cell.
    if chamber_names:
        profile_data["extra_chambers"] = " | ".join(chamber_names)

    if chamber_addresses:
        profile_data["extra_addresses"] = " | ".join(chamber_addresses)

    if visiting_times:
        profile_data["extra_visiting_hours"] = " | ".join(visiting_times)

    if phone_numbers:
        profile_data["appointments"] = " | ".join(phone_numbers)

    return profile_data


# =============================================================================
# STEP 4: READ THE MAIN PAGE THAT CONTAINS ALL DOCTORS
# =============================================================================

def read_one_doctor_card(doctor_card):
    """Read the basic information from one doctor card."""
    doctor = {
        # These two URLs are temporary.
        # They will not be included in the final CSV file.
        "image_url": None,
        "profile_url": None,

        # These values will appear in the CSV file.
        "title": None,
        "degree": None,
        "speciality": None,
        "experience": None,
        "designation": None,
        "workplace": None,
        "chamber": None,
        "address": None,
        "visiting_hour": None,
    }

    # Read the doctor's name.
    name_tag = doctor_card.find("h3", class_="title")
    doctor["title"] = get_text(name_tag)

    # Read the doctor's picture URL.
    image_tag = doctor_card.find("img")
    doctor["image_url"] = find_image_url(image_tag)

    # Read the doctor's profile URL.
    profile_link = doctor_card.find("a", class_="call-now")

    if profile_link and profile_link.get("href"):
        doctor["profile_url"] = urljoin(
            BASE_URL,
            profile_link["href"],
        )

    # Read degree, experience, specialty, designation, and workplace.
    doctor_information = doctor_card.find("ul", class_="doctor-info")

    if doctor_information is not None:
        for list_item in doctor_information.find_all("li"):
            label = list_item.get("title", "").strip().lower()
            value = get_text(list_item)

            if label == "degree":
                doctor["degree"] = value
            elif label == "experiences":
                doctor["experience"] = value
            elif label == "specialty":
                doctor["speciality"] = value
            elif label == "designation":
                doctor["designation"] = value
            elif label == "workplace":
                doctor["workplace"] = value

    # Read the main chamber, address, and visiting hour.
    chamber_information = doctor_card.find(
        "ul",
        class_="chamber-info",
    )

    if chamber_information is not None:
        for list_item in chamber_information.find_all("li"):
            label = list_item.get("title", "").strip().lower()
            value = get_text(list_item)

            if label == "chamber":
                doctor["chamber"] = value

            elif label == "address" and value:
                doctor["address"] = re.sub(
                    r"^Address:\s*",
                    "",
                    value,
                    flags=re.IGNORECASE,
                )

            elif label == "visiting hour" and value:
                doctor["visiting_hour"] = re.sub(
                    r"^Visiting Hour:\s*",
                    "",
                    value,
                    flags=re.IGNORECASE,
                )

    return doctor


def read_all_doctor_cards(html):
    """Find and read every doctor card on the main page."""
    page = BeautifulSoup(html, "lxml")

    doctor_list = page.select_one("#doctor-list-container")

    if doctor_list is None:
        raise RuntimeError(
            "The doctor list was not found. "
            "The website may have changed."
        )

    # The website now uses <div class="doctor">.
    # The old code incorrectly searched for <li class="doctor">.
    doctor_cards = doctor_list.select("div.doctor")

    if not doctor_cards:
        raise RuntimeError(
            "No doctors were found. "
            "Check PAGE_URL and the website structure."
        )

    doctors = []

    for doctor_card in doctor_cards:
        doctor = read_one_doctor_card(doctor_card)
        doctors.append(doctor)

    return doctors


# =============================================================================
# STEP 5: ADD PROFILE INFORMATION AND PICTURES
# =============================================================================

def complete_one_doctor(doctor):
    """Add profile details and a picture to one doctor's information."""
    completed_doctor = doctor.copy()

    try:
        profile_data = read_profile(
            completed_doctor["profile_url"]
        )
        completed_doctor.update(profile_data)

        completed_doctor["photo"] = download_image(
            completed_doctor["image_url"],
            completed_doctor["title"],
        )

    except Exception as error:
        # Keep the basic information even if one profile has a problem.
        print(
            f"Could not finish {completed_doctor['title']}: {error}"
        )

        completed_doctor.update(make_empty_profile())
        completed_doctor["photo"] = None

    # The final CSV must not contain website URLs.
    completed_doctor.pop("image_url", None)
    completed_doctor.pop("profile_url", None)

    return completed_doctor


# =============================================================================
# STEP 6: SAVE EVERYTHING
# =============================================================================

def save_results(doctors):
    """Save doctor information as CSV and pictures as ZIP."""
    # Giving pandas the column list prevents the old KeyError.
    table = pd.DataFrame(
        doctors,
        columns=CSV_COLUMNS,
    )

    # Remove repeated doctors with the same displayed name.
    table = table.drop_duplicates(
        subset=["title"],
        keep="first",
    )

    # Change empty values into the word null.
    table = table.fillna("null")

    # Save the CSV file.
    table.to_csv(
        CSV_NAME,
        index=False,
        encoding="utf-8-sig",
    )

    # Put the image folder inside a ZIP file.
    shutil.make_archive(
        IMAGE_FOLDER,
        "zip",
        IMAGE_FOLDER,
    )

    return table


# =============================================================================
# STEP 7: RUN ALL STEPS IN THE CORRECT ORDER
# =============================================================================

def main():
    """Run the complete program."""
    # Create the image folder if it does not already exist.
    os.makedirs(IMAGE_FOLDER, exist_ok=True)

    # Download the main doctor-list page.
    html = download_page(PAGE_URL)

    if not html:
        raise SystemExit(
            "The doctor page could not be downloaded. "
            "Check your internet connection and PAGE_URL."
        )

    # Read the basic information of every doctor.
    doctors = read_all_doctor_cards(html)
    print("Doctors Found:", len(doctors))

    # Work on several doctors together to finish faster.
    # executor.map also keeps the original doctor order.
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as worker:
        completed_doctors = list(
            worker.map(complete_one_doctor, doctors)
        )

    # Save the final files.
    table = save_results(completed_doctors)

    print(table.head())
    print("Total Doctors:", len(table))
    print("CSV Saved:", CSV_NAME)
    print("ZIP Saved:", IMAGE_FOLDER + ".zip")


# Run main() only when this file is started directly.
if __name__ == "__main__":
    main()

Doctors Found: 50
                                           photo  \
0        cardiac-surgeon/dr-samir-azam-sunny.jpg   
1  cardiac-surgeon/assoc-prof-dr-md-alauddin.jpg   
2  cardiac-surgeon/prof-dr-gm-mokbul-hossain.jpg   
3           cardiac-surgeon/dr-lutfor-rahman.jpg   
4          cardiac-surgeon/dr-jahangir-kabir.jpg   

                           title                        degree  \
0         Dr. Samir Azam (Sunny)  MBBS (RAJ), MS (CVTS), BSMMU   
1  Assoc. Prof. Dr. Md. Alauddin            MBBS, MS (CS & TS)   
2    Prof. Dr. GM Mokbul Hossain               MBBS, MS (CVTS)   
3              Dr. Lutfor Rahman                MBBS, MS (CTS)   
4             Dr. Jahangir Kabir               MBBS, MS (CVTS)   

                                          speciality experience  \
0               Cardiac, Vascular & Thoracic Surgeon       null   
1  Cardiac, Vascular & Thoracic Surgeon 20+ Years...       null   
2              Cardiac & Thoracic Surgery Specialist       null   
3   

In [8]:
import shutil
import os

# Specify the path to the folder you want to delete
folder_to_delete = 'anesthesiologist' # Replace with your folder path

if os.path.exists(folder_to_delete):
    try:
        shutil.rmtree(folder_to_delete)
        print(f"Folder '{folder_to_delete}' and its contents deleted successfully.")
    except OSError as e:
        print(f"Error: {e.filename} - {e.strerror}.")
else:
    print(f"Folder '{folder_to_delete}' does not exist.")

Folder 'anesthesiologist' and its contents deleted successfully.
